In [3]:
import numpy as np
import pandas as pd

# Data Extracting

In [4]:
df= pd.read_csv("Customer_Dataset.csv", sep=";")

In [5]:
df

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,...,Subscription Status,Shipping Type,Promo Code Used,Previous Purchases,Payment Method,Purchase Date,WeekdayNum,Weekday,Weekend,Churn
0,1,55,Male,Belt,Accessories,"46,9",Kentucky,L,Gray,Winter,...,Yes,Standard,0,14,Credit Card,7.01.2022,5,Friday,0,1
1,1,55,Male,Sweater,Clothing,"48,1",Kentucky,L,Green,Spring,...,Yes,Free Shipping,0,14,Credit Card,19.03.2022,6,Saturday,1,1
2,1,55,Male,Sweater,Clothing,"62,1",Kentucky,L,Green,Spring,...,Yes,Next Day Air,0,14,Debit Card,3.05.2022,2,Tuesday,0,1
3,1,55,Male,Belt,Accessories,"25,0",Kentucky,L,Turquoise,Summer,...,Yes,Free Shipping,0,14,Credit Card,7.05.2022,6,Saturday,1,1
4,1,55,Male,Sweater,Clothing,"89,8",Kentucky,L,Blue,Summer,...,Yes,2-Day Shipping,0,14,Google Pay,10.06.2022,5,Friday,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102766,3900,52,Female,Blouse,Clothing,"37,4",California,M,Brown,Winter,...,No,Standard,0,33,PayPal,1.12.2024,7,Sunday,1,0
102767,3900,52,Female,Blouse,Clothing,"66,0",California,M,Pink,Spring,...,No,Standard,0,33,Debit Card,20.12.2024,5,Friday,0,0
102768,3900,52,Female,Blouse,Clothing,"39,8",California,M,Maroon,Fall,...,No,Free Shipping,1,33,Debit Card,21.12.2024,6,Saturday,1,0
102769,3900,52,Female,Sneakers,Footwear,"90,7",California,M,Orange,Fall,...,No,2-Day Shipping,0,33,PayPal,22.12.2022,4,Thursday,0,0


In [6]:
df.shape

(102771, 21)

In [7]:
df.columns

Index(['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category',
       'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season',
       'Review Rating', 'Subscription Status', 'Shipping Type',
       'Promo Code Used', 'Previous Purchases', 'Payment Method',
       'Purchase Date', 'WeekdayNum', 'Weekday', 'Weekend', 'Churn'],
      dtype='str')

# Data Transformation

In [8]:
df.dtypes

Customer ID              int64
Age                      int64
Gender                     str
Item Purchased             str
Category                   str
Purchase Amount (USD)      str
Location                   str
Size                       str
Color                      str
Season                     str
Review Rating              str
Subscription Status        str
Shipping Type              str
Promo Code Used          int64
Previous Purchases       int64
Payment Method             str
Purchase Date              str
WeekdayNum               int64
Weekday                    str
Weekend                  int64
Churn                    int64
dtype: object

By checking data types i found 3 mismathced data types 
- purchage amount 
- Review Rating
- Purchage date

In [9]:
df.isnull().sum().sum()

np.int64(0)

There is no Missing values in a dataset

In [10]:
df.duplicated().sum()

np.int64(0)

And also there is no any Duplicate Values

# Changing Data Types

In [11]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

In [12]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_(usd)', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'promo_code_used', 'previous_purchases', 'payment_method',
       'purchase_date', 'weekdaynum', 'weekday', 'weekend', 'churn'],
      dtype='str')

In [13]:
df["review_rating"] = (
    df["review_rating"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float))

In [14]:
df["purchase_date"] = pd.to_datetime(
    df["purchase_date"],
    format="%d.%m.%Y")

In [15]:
df["purchase_amount"] = (df["purchase_amount_(usd)"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float))

df.drop(columns=["purchase_amount_(usd)"], inplace=True)

In [16]:
df.dtypes

customer_id                     int64
age                             int64
gender                            str
item_purchased                    str
category                          str
location                          str
size                              str
color                             str
season                            str
review_rating                 float64
subscription_status               str
shipping_type                     str
promo_code_used                 int64
previous_purchases              int64
payment_method                    str
purchase_date          datetime64[us]
weekdaynum                      int64
weekday                           str
weekend                         int64
churn                           int64
purchase_amount               float64
dtype: object

# Chacking incoorect Values

In [17]:
df["gender"].unique()

<ArrowStringArray>
['Male', 'Female']
Length: 2, dtype: str

In [18]:
df["category"].unique()

<ArrowStringArray>
['Accessories', 'Clothing', 'Outerwear', 'Footwear']
Length: 4, dtype: str

In [19]:
df["season"].unique()

<ArrowStringArray>
['Winter', 'Spring', 'Summer', 'Fall']
Length: 4, dtype: str

In [20]:
df["subscription_status"].unique()

<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str

In [21]:
df["promo_code_used"].unique()

array([0, 1])

In [22]:
df["weekend"].unique()

array([0, 1])

In [23]:
df["churn"].unique()

array([1, 0])

# Feature Engineering

In [24]:
# create a column age group

labels=['Young', 'Adult','Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4,labels=labels)

In [25]:
df[['age', 'age_group']].head(20)

,age,age_group
0,55,Middle-aged
1,55,Middle-aged
2,55,Middle-aged
3,55,Middle-aged
4,55,Middle-aged
5,55,Middle-aged
6,55,Middle-aged
7,55,Middle-aged
8,55,Middle-aged
9,55,Middle-aged


Recency

In [26]:
max_date = df["purchase_date"].max()

df["last_purchase_date"] = (
    df.groupby("customer_id")["purchase_date"].transform("max"))

df["recency"] = (
    max_date - df["last_purchase_date"]).dt.days

Frequency

In [27]:
df["frequency"] = (
    df.groupby("customer_id")["customer_id"]
      .transform("count"))

Monetary

In [28]:
df["monetary"] = (
    df.groupby("customer_id")["purchase_amount"]
      .transform("sum"))

In [29]:
df[ ["customer_id",
     "purchase_date",
     "last_purchase_date",
     "recency",
     "frequency",
     "monetary"]].head(20)

,customer_id,purchase_date,last_purchase_date,recency,frequency,monetary
0,1,2022-01-07,2024-07-09,175,15,857.9
1,1,2022-03-19,2024-07-09,175,15,857.9
2,1,2022-05-03,2024-07-09,175,15,857.9
3,1,2022-05-07,2024-07-09,175,15,857.9
4,1,2022-06-10,2024-07-09,175,15,857.9
5,1,2022-08-28,2024-07-09,175,15,857.9
6,1,2023-01-01,2024-07-09,175,15,857.9
7,1,2023-02-11,2024-07-09,175,15,857.9
8,1,2023-03-04,2024-07-09,175,15,857.9
9,1,2023-03-05,2024-07-09,175,15,857.9


In [30]:
df[["customer_id"]]

,customer_id
0,1
1,1
2,1
3,1
4,1
...,...
102766,3900
102767,3900
102768,3900
102769,3900


In [31]:
import pymysql

In [32]:
! pip install mysql sqlalchemy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:

from sqlalchemy import create_engine
host="localhost"
user="root"
password="root"
port="3306"
database="customer"

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{database}")

table_name = "customer_dataset"
df.to_sql(table_name, engine, if_exists="replace", index=False)

pd.read_sql(f"SELECT * FROM {table_name} LIMIT 10", engine)


,customer_id,age,gender,item_purchased,category,location,size,color,season,review_rating,...,weekdaynum,weekday,weekend,churn,purchase_amount,age_group,last_purchase_date,recency,frequency,monetary
0,1,55,Male,Belt,Accessories,Kentucky,L,Gray,Winter,5.0,...,5,Friday,0,1,46.9,Middle-aged,2024-07-09,175,15,857.9
1,1,55,Male,Sweater,Clothing,Kentucky,L,Green,Spring,1.0,...,6,Saturday,1,1,48.1,Middle-aged,2024-07-09,175,15,857.9
2,1,55,Male,Sweater,Clothing,Kentucky,L,Green,Spring,5.0,...,2,Tuesday,0,1,62.1,Middle-aged,2024-07-09,175,15,857.9
3,1,55,Male,Belt,Accessories,Kentucky,L,Turquoise,Summer,5.0,...,6,Saturday,1,1,25.0,Middle-aged,2024-07-09,175,15,857.9
4,1,55,Male,Sweater,Clothing,Kentucky,L,Blue,Summer,5.0,...,5,Friday,0,1,89.8,Middle-aged,2024-07-09,175,15,857.9
5,1,55,Male,Sweater,Clothing,Kentucky,L,Beige,Fall,2.5,...,7,Sunday,1,1,70.3,Middle-aged,2024-07-09,175,15,857.9
6,1,55,Male,Sweater,Clothing,Kentucky,L,Brown,Winter,5.0,...,7,Sunday,1,1,84.6,Middle-aged,2024-07-09,175,15,857.9
7,1,55,Male,Sweater,Clothing,Kentucky,L,Turquoise,Spring,4.5,...,6,Saturday,1,1,27.6,Middle-aged,2024-07-09,175,15,857.9
8,1,55,Male,Coat,Outerwear,Kentucky,L,Red,Winter,5.0,...,6,Saturday,1,1,44.6,Middle-aged,2024-07-09,175,15,857.9
9,1,55,Male,Sweater,Clothing,Kentucky,L,White,Spring,5.0,...,7,Sunday,1,1,22.3,Middle-aged,2024-07-09,175,15,857.9
